In [ ]:
import json

from abc import ABC, abstractmethod
from enum import Enum
from pydantic import BaseModel, Field

In [ ]:
class Room(ABC):
    def __init__(self, room_number: int, name: str, base_price: str, rooms_count: int = 2):
        self.room_number = room_number
        self.name = name
        self.base_price = base_price
        self.rooms_count = rooms_count
    
    @abstractmethod
    def get_amenities(self) -> list[str]:
        pass
    
    @abstractmethod
    def get_price(self) -> float:
        pass
    
    def get_details(self):
        return f"{self.name} room. Price: {self.get_price()}. Amenities: {self.get_amenities()}"
    
class SuiteRoom(Room):
    def get_amenities(self) -> list[str]:
        return ["TV", "Coffee Machine", "AC"]
    
    def get_price(self):
        return self.base_price * 2
    
class DeluxeRoom(Room):
    def get_amenities(self) -> list[str]:
        return ["TV", "Coffee Machine", "AC", "Pool", "Hair Dryer"]
    
    def get_price(self) -> float:
        return self.base_price * 3

In [ ]:
class PaymentMethod(Enum):
    NETBANKING = 1
    CREDITCARD = 2
    CASH = 3

class Guest:
    def __init__(self, name: str, mobile_number: str, payment_method: PaymentMethod):
        self.name = name
        self.mobile_number = mobile_number
        self.__payment_method = payment_method
        
    def get_user_details(self):
        return f"The user name is {self.name} and the mobile number is {self.mobile_number}"
    
    def get_user_payment_method(self) -> PaymentMethod:
        return self.__payment_method

In [ ]:
class BookingViewModel(BaseModel):
    room_number: int
    guest_name: str
    guest_mobile_number: str
    check_in: str

booking_db: list[BookingViewModel] = []

In [ ]:
class Booking:
    def __init__(self, room: Room, guest: Guest, check_in: str):
        self.room = room
        self.guest = guest
        self.check_in = check_in
        
    def __is_room_availabel(self) -> bool:
        booking_results = [booking for booking in booking_db if booking.room_number == self.room.room_number and booking.check_in == self.check_in]
        if len(booking_results) >= self.room.rooms_count:
            return False
        else:
            return True
        
    def book(self) -> str:
        if self.__is_room_availabel():
            booking_db.append(
                BookingViewModel(
                    room_number=self.room.room_number,
                    guest_name=self.guest.name,
                    guest_mobile_number=self.guest.mobile_number,
                    check_in=self.check_in
                )
            )
            return "Room booked successfully!"
        else:
            return "Room not availabel!"
        
    def get_bookings(self) -> list[BookingViewModel]:
        return booking_db

In [ ]:
rooms: list[Room] = []
rooms.append(SuiteRoom(room_number=101, name="Ostrich Suite Room", base_price=100))
rooms.append(DeluxeRoom(room_number=201, name="Ostrich Deluxe Room", base_price=100))

guest = Guest(name="Chandan", mobile_number="0123456789", payment_method=PaymentMethod.CASH)

booking = Booking(room=rooms[0], guest=guest, check_in="15-12-2026")
print(booking.book())


In [ ]:
all_bookings = json.dumps([booking.model_dump() for booking in booking.get_bookings()], indent=4)
print(all_bookings)